# Visualize the shared regularizer transformation strength

Choose one `TRANSFORMATION_STRENGTH` between 0 and 1 below. The notebook selects the same fixed source anchors as `Experiment.adapt()`, finds each family's strongest non-redundant and non-degenerate modification, and resolves one intensity per transformation. Images are columns and transformations are rows. The final cell recomputes every achieved RMS displacement $s_k$ over all anchors.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import torch


def find_repo_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the repository root.")


REPO_ROOT = find_repo_root(Path.cwd())
SOURCE_DIR = REPO_ROOT / "src"
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

from domain_knowledge_analysis import utils

print(f"Repository: {REPO_ROOT}")

## Setting

Change `TRANSFORMATION_STRENGTH` and rerun the cells below. The shared $s$ is the fraction of each transformation's own maximum anchor RMS: values near 0 are mild and $s=1$ is the earliest strongest valid modification. Configured intensities determine direction.

In [ ]:
CONFIG_PATH = REPO_ROOT / "config" / "lora_vae_mnist.yaml"
TRANSFORMATION_STRENGTH = 0.5
NUMBER_OF_IMAGES = 5
DEVICE = torch.device("cpu")

In [ ]:
config = utils.load_config(CONFIG_PATH)
config["lora"]["regularizer"].pop("target_input_rms", None)
config["lora"]["regularizer"]["transformation_strength"] = TRANSFORMATION_STRENGTH
dataset_name = config["dataset"]["name"]
anchor_points = utils.select_anchor_points(config, dataset_name, DEVICE)
transformations, _ = utils.create_regularizer_transformations(config, DEVICE, anchor_points)
calibration = config["lora"]["regularizer"]["calibration"]
display_indices = torch.linspace(0, len(anchor_points) - 1, NUMBER_OF_IMAGES).long()
display_images = anchor_points[display_indices]

print(f"Anchors: {len(anchor_points)}")
print(f"Transformation strength: {TRANSFORMATION_STRENGTH:g}")

In [ ]:
def show_image(axis, image):
    image = image.detach().cpu()
    axis.imshow(image[0].numpy(), cmap="gray", vmin=0, vmax=1, interpolation="nearest")
    axis.set_xticks([])
    axis.set_yticks([])


rows = len(transformations) + 1
figure, axes = plt.subplots(rows, NUMBER_OF_IMAGES, figsize=(2.1 * NUMBER_OF_IMAGES, 1.8 * rows), squeeze=False)

for column, image in enumerate(display_images):
    show_image(axes[0, column], image)
    axes[0, column].set_title(f"Image {column + 1}")
axes[0, 0].set_ylabel("Original", rotation=0, ha="right", va="center", labelpad=12)

for row, (transformation, metadata) in enumerate(zip(transformations, calibration["transformations"]), start=1):
    transformed_images = transformation(display_images)
    for column, image in enumerate(transformed_images):
        show_image(axes[row, column], image)
    axes[row, 0].set_ylabel(metadata["name"].replace("_", " "), rotation=0, ha="right", va="center", labelpad=12)

figure.suptitle(f"Regularizer transformations at relative strength s = {TRANSFORMATION_STRENGTH:g}", fontsize=14)
figure.tight_layout(rect=[0, 0, 1, 0.985])
plt.show()

## Achieved input RMS $s_k$

These values are recomputed from the transformed pixels over every fixed anchor, channel and pixel.

In [ ]:
batch_size = config["lora"]["regularizer"].get("calibration_batch_size", 100)
print(f"{'transformation':30} {'intensity':>12} {'max intensity':>14} {'max s_k':>12} {'computed s_k':>16} {'fraction':>12}")
print("-" * 104)

for transformation, metadata in zip(transformations, calibration["transformations"]):
    achieved_rms = utils.measure_input_rms(transformation, anchor_points, batch_size)
    relative_strength = achieved_rms / metadata["maximum_input_rms"]
    print(f"{metadata['name']:30} {metadata['resolved_intensity']:12.6g} {metadata['maximum_intensity']:14.6g} {metadata['maximum_input_rms']:12.6f} {achieved_rms:16.8f} {relative_strength:12.3f}")